# Question 28: Build a Complete LLM Inference Engine (SOLUTION)

### Difficulty: Expert

### Problem Statement

In this capstone exercise, you will build a **minimal but complete** LLM inference engine that integrates several components you have learned:

1. **A simple transformer model** (2-layer, with self-attention and feed-forward layers)
2. **KV cache** for efficient autoregressive generation
3. **Top-p (nucleus) sampling** for controllable text generation
4. **Basic batching** to handle multiple concurrent requests

The engine accepts text prompts (using a simple character-level tokenizer), generates completions, and supports multiple concurrent requests.

This mirrors the core of what production inference engines (vLLM, TGI, TensorRT-LLM) do, stripped down to its essentials.

---

### Requirements

1. **`SimpleTokenizer`**: Character-level tokenizer.
   - `encode(text) -> List[int]`: Convert text to token IDs.
   - `decode(ids) -> str`: Convert token IDs back to text.
   - Reserve ID 0 for `<pad>` and ID 1 for `<eos>`.

2. **`MiniTransformer(nn.Module)`**: A 2-layer transformer.
   - Token embedding + positional embedding.
   - Each layer: multi-head self-attention (with KV cache support) + feed-forward network.
   - Layer normalization (pre-norm style).
   - Output projection to vocabulary logits.

3. **`InferenceEngine`**: Ties everything together.
   - `generate(prompts, max_new_tokens, temperature, top_p)`: Accept a list of text prompts, tokenize, generate completions, decode back to text.
   - Uses KV cache internally for efficient generation.
   - Implements top-p sampling.

4. **Validation**: Generate text from multiple prompts simultaneously, showing the engine handles them correctly.

---

### Constraints

- Use only PyTorch operations.
- The transformer must use pre-norm (LayerNorm before attention/FFN, not after).
- Top-p sampling: sort probabilities descending, compute cumulative sum, zero out tokens beyond the cumulative threshold `p`, renormalize, then sample.
- The model does NOT need to produce coherent text (it is untrained). The goal is correct architecture and inference logic.

---

### Company Tags

Perplexity, Together AI, Anyscale, Fireworks AI

---

<details>
  <summary>Hint</summary>

  - For the tokenizer, use `ord(c) + 2` as the token ID for character `c` (offset by 2 to reserve 0 and 1 for special tokens). For decoding, `chr(id - 2)`.
  - For KV cache in the transformer: each layer gets its own KVCache. Pass a list of caches (one per layer) through the forward pass.
  - For top-p sampling: `sorted_probs, sorted_indices = torch.sort(probs, descending=True)`. Then `cumsum = torch.cumsum(sorted_probs, dim=-1)`. Mask out where `cumsum - sorted_probs >= top_p`.
  - For batching: pad all prompts to the same length before passing to the model. Use attention masking to ignore padding.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Optional, Tuple

In [ ]:
# Configuration
torch.manual_seed(42)

d_model = 64
num_heads = 4
d_ff = 128              # Feed-forward hidden dimension
num_layers = 2
max_seq_len = 256
temperature = 0.8
top_p = 0.9

device = "cpu"
print(f"Config: d_model={d_model}, heads={num_heads}, layers={num_layers}, d_ff={d_ff}")

In [ ]:
class SimpleTokenizer:
    """
    Character-level tokenizer.
    
    Special tokens:
        0 = <pad>
        1 = <eos>
    Regular characters: ord(c) + 2
    """
    
    def __init__(self):
        self.pad_id = 0
        self.eos_id = 1
        # Cover all printable ASCII (32-126) + 2 special tokens
        self.vocab_size = 128 + 2  # 130 total
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        return [ord(c) + 2 for c in text]
    
    def decode(self, ids):
        """Convert list of token IDs back to text. Skip pad/eos."""
        chars = []
        for token_id in ids:
            if token_id <= 1:  # Skip pad and eos
                continue
            chars.append(chr(token_id - 2))
        return ''.join(chars)


# Quick test
tok = SimpleTokenizer()
assert tok.decode(tok.encode("abc")) == "abc"
print(f"SimpleTokenizer: vocab_size={tok.vocab_size}, pad_id={tok.pad_id}, eos_id={tok.eos_id}")

In [ ]:
class KVCache:
    """KV Cache for a single attention layer."""
    
    def __init__(self):
        self.k = None
        self.v = None
    
    def update(self, new_k, new_v):
        """Append new K/V and return full cached K/V."""
        if self.k is None:
            self.k = new_k
            self.v = new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
        return self.k, self.v
    
    @property
    def seq_len(self):
        """Return the current cached sequence length."""
        return 0 if self.k is None else self.k.shape[2]
    
    def reset(self):
        self.k = None
        self.v = None


class MultiHeadAttention(nn.Module):
    """Multi-head attention with KV cache support."""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, kv_cache=None):
        """
        Args:
            x: (B, S, d_model)
            kv_cache: Optional KVCache
        Returns:
            output: (B, S, d_model)
        """
        B, S, _ = x.shape
        
        q = self.W_q(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        k = self.W_k(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        
        if kv_cache is not None:
            k, v = kv_cache.update(k, v)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        # Apply causal mask only when not using cache (full sequence mode)
        if kv_cache is None:
            causal_mask = torch.tril(torch.ones(S, S, device=x.device)).bool()
            scores = scores.masked_fill(~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        
        q_len = q.shape[2]
        output = output.transpose(1, 2).contiguous().view(B, q_len, self.d_model)
        return self.W_o(output)


class FeedForward(nn.Module):
    """Simple feed-forward network: Linear -> GELU -> Linear."""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class TransformerBlock(nn.Module):
    """Single transformer block: pre-norm attention + pre-norm FFN."""
    
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    
    def forward(self, x, kv_cache=None):
        """
        Pre-norm transformer block:
            x = x + Attention(LayerNorm(x))
            x = x + FFN(LayerNorm(x))
        """
        x = x + self.attn(self.ln1(x), kv_cache=kv_cache)
        x = x + self.ffn(self.ln2(x))
        return x


class MiniTransformer(nn.Module):
    """
    Minimal 2-layer transformer for text generation.
    
    Components:
    - Token embedding + positional embedding
    - N transformer blocks (each with attention + FFN)
    - Final layer norm
    - Output projection to vocab logits
    """
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        
        self.final_norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, input_ids, kv_caches=None):
        """
        Args:
            input_ids: (B, S) token indices
            kv_caches: Optional list of KVCache (one per layer)
        Returns:
            logits: (B, S, vocab_size)
        """
        B, S = input_ids.shape
        
        # Determine position offset from cache
        if kv_caches is not None and kv_caches[0].seq_len > 0:
            pos_offset = kv_caches[0].seq_len - S  # Cache already has some positions
            # Actually, during generation with cache, the cache was updated
            # BEFORE this point for the previous token. But we compute positions
            # for the current input tokens.
            # Wait -- the cache update happens inside attention, so at this point
            # the cache has NOT yet been updated with the current tokens.
            # Let me reconsider: kv_cache.update() is called inside MultiHeadAttention.forward().
            # So at the start of MiniTransformer.forward(), the cache has positions 0..t-1.
            # The current input has positions t..t+S-1.
            pos_offset = kv_caches[0].seq_len
        else:
            pos_offset = 0
        
        # Token + positional embeddings
        positions = torch.arange(pos_offset, pos_offset + S, device=input_ids.device)
        x = self.token_emb(input_ids) + self.pos_emb(positions).unsqueeze(0)
        
        # Pass through transformer blocks
        for i, block in enumerate(self.blocks):
            cache = kv_caches[i] if kv_caches is not None else None
            x = block(x, kv_cache=cache)
        
        # Final norm + output projection
        x = self.final_norm(x)
        logits = self.output_proj(x)
        
        return logits


print("Model components defined.")

In [ ]:
def top_p_sample(logits, temperature=1.0, top_p=0.9):
    """
    Top-p (nucleus) sampling.
    
    Args:
        logits: (B, vocab_size) raw logits for next token
        temperature: Sampling temperature
        top_p: Cumulative probability threshold
    
    Returns:
        sampled_tokens: (B, 1) sampled token IDs
    """
    # 1. Apply temperature
    scaled_logits = logits / temperature
    
    # 2. Softmax
    probs = F.softmax(scaled_logits, dim=-1)  # (B, vocab_size)
    
    # 3. Sort descending
    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
    
    # 4. Cumulative sum
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    
    # 5. Create mask: keep tokens where cumsum (excluding current) < top_p
    # This means we keep the smallest set of tokens whose cumulative prob >= top_p
    mask = (cumsum - sorted_probs) >= top_p  # True for tokens to REMOVE
    
    # 6. Zero out masked tokens
    sorted_probs[mask] = 0.0
    
    # 7. Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    
    # 8. Sample from the filtered distribution
    sampled_sorted_idx = torch.multinomial(sorted_probs, num_samples=1)  # (B, 1)
    
    # 9. Map back to original token IDs
    sampled_tokens = sorted_indices.gather(dim=-1, index=sampled_sorted_idx)  # (B, 1)
    
    return sampled_tokens


class InferenceEngine:
    """
    Complete inference engine tying together:
    - Tokenizer
    - Transformer model with KV cache
    - Top-p sampling
    - Basic batching
    """
    
    def __init__(self, model, tokenizer, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
    
    def generate(self, prompts, max_new_tokens=50, temperature=0.8, top_p=0.9):
        """
        Generate completions for a list of text prompts.
        
        Args:
            prompts: List of text strings
            max_new_tokens: Maximum tokens to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
        
        Returns:
            List of generated text strings (prompt + completion)
        """
        batch_size = len(prompts)
        
        # 1. Tokenize all prompts
        encoded = [self.tokenizer.encode(p) for p in prompts]
        prompt_lengths = [len(e) for e in encoded]
        max_prompt_len = max(prompt_lengths)
        
        # 2. Right-pad to same length
        padded = []
        for e in encoded:
            padded.append(e + [self.tokenizer.pad_id] * (max_prompt_len - len(e)))
        
        input_ids = torch.tensor(padded, device=self.device)  # (B, max_prompt_len)
        
        # 3. Create KV caches (one per layer)
        kv_caches = [KVCache() for _ in range(self.model.num_layers)]
        
        # 4. Prefill: run model on full prompt to populate KV cache
        logits = self.model(input_ids, kv_caches=kv_caches)  # (B, max_prompt_len, vocab)
        
        # Get the logits for the last real token of each prompt
        # For simplicity (since we right-pad), use the last position
        next_logits = logits[:, -1, :]  # (B, vocab_size)
        
        # 5. Decode loop
        all_generated = [[] for _ in range(batch_size)]
        done = [False] * batch_size
        
        for step in range(max_new_tokens):
            # Sample next token using top-p
            next_tokens = top_p_sample(next_logits, temperature=temperature, top_p=top_p)  # (B, 1)
            
            # Record generated tokens
            for i in range(batch_size):
                if not done[i]:
                    token = next_tokens[i].item()
                    if token == self.tokenizer.eos_id:
                        done[i] = True
                    else:
                        all_generated[i].append(token)
            
            # Check if all done
            if all(done):
                break
            
            # Feed new tokens through model with KV cache
            logits = self.model(next_tokens, kv_caches=kv_caches)  # (B, 1, vocab)
            next_logits = logits[:, -1, :]  # (B, vocab_size)
        
        # 6. Decode back to text
        outputs = []
        for i in range(batch_size):
            generated_text = self.tokenizer.decode(all_generated[i])
            outputs.append(prompts[i] + generated_text)
        
        return outputs


print("InferenceEngine defined.")

In [ ]:
# ============================================================
# Validation: Generate from multiple prompts
# ============================================================

torch.manual_seed(42)

# Initialize tokenizer
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")

# Test tokenizer roundtrip
test_text = "Hello, World!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Tokenizer roundtrip: '{test_text}' -> {encoded} -> '{decoded}'")
assert decoded == test_text, "Tokenizer roundtrip failed!"
print()

# Initialize model and engine
model = MiniTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    max_seq_len=max_seq_len
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print()

engine = InferenceEngine(model, tokenizer, device=device)

# Generate from multiple prompts
prompts = [
    "The quick brown ",
    "In a galaxy ",
    "def fibonacci(",
    "Once upon a ",
]

print("Generating completions...")
print("=" * 60)

with torch.no_grad():
    outputs = engine.generate(
        prompts,
        max_new_tokens=40,
        temperature=temperature,
        top_p=top_p
    )

for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    generated_part = output[len(prompt):]
    print(f"\nPrompt {i}: '{prompt}'")
    print(f"Generated: '{generated_part}'")
    print(f"Full: '{output}'")

print("\n" + "=" * 60)
print("Inference engine validation complete.")
print("(Note: output is random since the model is untrained.)")